<a href="https://colab.research.google.com/github/yato-hash/prototype_solution_challenge/blob/main/Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pytesseract opencv-python pillow moviepy openai-whisper
!apt-get install tesseract-ocr -y

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 41.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=4472f77fd75f15f380ee2b31a39cb7551e876d8295add6c9d6234fc42a55f221
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [11]:
from PIL import Image
import pytesseract
import cv2
import numpy as np
import whisper
from moviepy.editor import VideoFileClip
from google.colab import files

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [12]:
def fake_text_score(text):
    text = text.lower()

    keywords = ["breaking", "shock", "viral", "100", "guarantee"]

    score = 0
    for word in keywords:
        if word in text:
            score += 1

    return min(score * 20, 100)

In [13]:
# Upload image
uploaded = files.upload()

# Get filename
img_name = list(uploaded.keys())[0]

# Read + preprocess
img_cv = cv2.imread(img_name)
gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

# OCR
text_img = pytesseract.image_to_string(thresh)

print("Extracted Text:\n", text_img)

# Fake score
score_img = fake_text_score(text_img)
print("Fake Score (Image):", score_img)

Saving Prototype.png to Prototype (1).png
Extracted Text:
 BREAKING [RSME
memes

Fake Score (Image): 20


In [15]:
# Upload audio file
uploaded = files.upload()
audio_name = list(uploaded.keys())[0]

# Load whisper model
model = whisper.load_model("base")

# Transcribe
result = model.transcribe(audio_name)
text_audio = result["text"]

print("Extracted Speech:\n", text_audio)

# Fake score
score_audio = fake_text_score(text_audio)
print("Fake Score (Audio):", score_audio)

Saving Prototype.mp3 to Prototype.mp3


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 122MiB/s]
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Extracted Speech:
  Breaking news. This shocking viral update claims a 100% guaranteed cure.
Fake Score (Audio): 100


In [19]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from moviepy.editor import ImageSequenceClip, AudioFileClip
import textwrap

# Install font (run once)
!apt-get install fonts-dejavu-core -y

width, height = 720, 1280

# 🔥 MATCHED WITH YOUR AUDIO TEXT
scenes = [
    "BREAKING NEWS",
    "This shocking viral update",
    "claims a 100 percent guaranteed cure"
]

def create_scene(text, color):
    frames = []
    for _ in range(40):  # longer duration per scene
        img = Image.new("RGB", (width, height), color=color)
        draw = ImageDraw.Draw(img)

        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 60)
        except:
            font = ImageFont.load_default()

        lines = textwrap.wrap(text, width=20)
        y = 400

        for line in lines:
            w, h = draw.textbbox((0, 0), line, font=font)[2:]
            draw.text(((width - w) // 2, y), line, fill="white", font=font)
            y += h + 20

        frames.append(np.array(img))
    return frames

colors = [(80,0,0),(20,20,20),(80,0,0)]

frames = []
for t, c in zip(scenes, colors):
    frames += create_scene(t, c)

clip = ImageSequenceClip(frames, fps=24)

# 🔊 USE YOUR FILE (CASE-SENSITIVE)
audio = AudioFileClip("Prototype.mp3")
clip = clip.set_audio(audio)

clip.write_videofile("final_video.mp4", fps=24)

video_name = "final_video.mp4"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-dejavu-core is already the newest version (2.37-2build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Moviepy - Building video final_video.mp4.
MoviePy - Writing audio in final_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video final_video.mp4



Moviepy - Done !
Moviepy - video ready final_video.mp4


In [20]:
# Use generated video directly
clip = VideoFileClip(video_name)
clip.audio.write_audiofile("temp_audio.mp3")

# Transcribe
result = model.transcribe("temp_audio.mp3")
text_video = result["text"]

print("Extracted Video Speech:\n", text_video)

# Fake score
score_video = fake_text_score(text_video)
print("Fake Score (Video):", score_video)

MoviePy - Writing audio in temp_audio.mp3


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



MoviePy - Done.
Extracted Video Speech:
  Breaking news. This shocking viral update claims a 100% guaranteed cure.
Fake Score (Video): 100


In [21]:
def final_decision(scores):
    avg = sum(scores) / len(scores)

    if avg > 60:
        return "Likely FAKE"
    elif avg > 30:
        return "Suspicious"
    else:
        return "Likely REAL"

scores = []

# Add available scores
try: scores.append(score_img)
except: pass

try: scores.append(score_audio)
except: pass

try: scores.append(score_video)
except: pass

if scores:
    print("\nFINAL RESULT:", final_decision(scores))


FINAL RESULT: Likely FAKE
